In [1]:
pip install youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [27]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_voyageai import VoyageAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate


In [22]:
video_id = "Gfr50f6ZBvo"

try:
    ytt_api = YouTubeTranscriptApi()
    fetched_list= ytt_api.fetch(video_id, languages=["en"])
    transcript=" ".join([snippet.text for snippet in fetched_list][0:100])
    print(transcript)
except TranscriptsDisabled:
    print("Transcripts are disabled for this video")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
texts = splitter.create_documents([transcript])

In [24]:
embeddings = VoyageAIEmbeddings(
    voyage_api_key=os.getenv("VOYAGE_API_KEY"), model="voyage-law-2"
)
vector_store = FAISS.from_documents(texts, embeddings)

vector_store.index_to_docstore_id


{0: '73afcd13-8621-46b1-bdc0-efa93b10e1f4',
 1: '40e95daa-98a3-42f9-a7a5-5cfdd73b13af',
 2: '25e3f009-7478-4665-9a95-c183890c5252',
 3: '8456d5d7-ae16-4333-8deb-22c692bad0fb',
 4: '6969d128-ffa3-4739-bbb1-736d4cba3f87'}

In [25]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={"k": 4})

In [26]:
retriever.invoke("What does this video talk about?")

[Document(id='40e95daa-98a3-42f9-a7a5-5cfdd73b13af', metadata={}, page_content="out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough to interview you well i'll be impressed if if you were i'd be impressed by myself if you were i don't think we're quite up to that yet but uh maybe you're from the future lex if you did would you tell me is that is that a good thing to tell a language model that's tasked with interviewing that it is in fact um ai maybe we're in a kind of meta turing test uh probably probably it would be a good idea not to tell you so it doesn't change your behavior right this is a kind of heisenberg uncertainty principle situation if i told you you behave differently yeah maybe that's what's happening with us of course this is a benchmark from the future where they replay 2022 as a year before ais were good enough yet and now we 

In [ ]:
llm =  ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="openai/gpt-oss-20b:free",
)

parser = StrOutputParser()

prompt_template = PromptTemplate(
    input_variables=["question"],
    template="""
    <s>[INST] You are a helpful assistant.

    Answer the question based on the transcript provided below.

    Transcript: {context}

    Question: {question}

    if you don't have enough context to answer the question, say i don't have enough context.

    Answer: [/INST]"""
)

In [39]:
question ="Is the topic of aliens discussed in this video? if not what is the topic discussed"

retriever_docs = retriever.invoke(question)

In [ ]:

context = "\n".join([doc.page_content for doc in retrieved_docs])

chain = prompt_template|llm|parser

try:
    answer = chain.invoke({"question": question, "context": context})
except Exception as e:
    answer = str(e)

print(answer)


No. The video does not discuss aliens.  
The conversation centers on artificial intelligence, specifically Demis Hassabis’s work at DeepMind, the nature of the Turing test, AI benchmarks, and what constitutes human‑level or superior AI performance.


In [48]:
question ="Is the topic of aliens discussed in this video? if not what is the topic discussed"

def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough(),
})

chain = parallel_chain | prompt_template | llm | parser

try:
    answer = chain.invoke(question)
except Exception as e:
    answer = str(e)

print(answer)

**Answer:**  
No, the topic of aliens is not discussed in this video. The conversation centers on artificial intelligence, the Turing test, and how AI systems are evaluated for human‑level performance.
